# Restormer 로 잡음 제거

앞단 필터 없이 **잡음 영상을 그대로 입력**으로 받아 원본 영상을 복원한다.

비교 기준 (팀원 B, 필터 없는 DnCNN):

| 잡음 종류 | PSNR | SSIM |
|---|---|---|
| gaussian | 34.413 | 0.9180 |
| rician | 30.529 | 0.8900 |
| uniform | 34.213 | 0.9477 |
| salt_and_pepper | 38.505 | 0.9869 |
| **ALL** | **34.415** | **0.9356** |

평가 지표(PSNR / SSIM)와 잡음 생성기는 기존 DnCNN 노트북과 **완전히 동일한 것**을 쓴다.
그래야 숫자가 직접 비교된다.


## 1. Google Drive 마운트 + 데이터 준비

In [9]:
# 입력 데이터 -> /content (VM 로컬 디스크, 빠름 / 세션 종료 시 삭제)
# 체크포인트   -> Drive (영속)
from pathlib import Path

import torch
from google.colab import drive

drive.mount("/content/drive/")

# TODO: 본인 Drive 경로에 맞게 수정
DRIVE_ROOT = Path("/content/drive/MyDrive/project5")
DATASET_ZIP = DRIVE_ROOT / "dataset.zip"

LOCAL_ROOT = Path("/content/work")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(f"Drive 경로가 없다: {DRIVE_ROOT}")

REQUIRED = ["train", "val", "test_label", "test_noise_only"]


def find_data_root(base: Path) -> Path | None:
    """REQUIRED 하위 폴더를 모두 가진 디렉터리를 찾는다 (zip 구조에 무관하게)."""
    for cand in [base, *(d for d in base.iterdir() if d.is_dir())]:
        if all((cand / name).is_dir() for name in REQUIRED):
            return cand
    return None


if find_data_root(LOCAL_ROOT) is None and DATASET_ZIP.exists():
    !cp "{DATASET_ZIP}" /content/dataset.zip
    !unzip -q -o /content/dataset.zip -d "{LOCAL_ROOT}"

LOCAL_DATA = find_data_root(LOCAL_ROOT)
if LOCAL_DATA is None:
    raise FileNotFoundError(f"데이터 폴더를 찾지 못했다: {LOCAL_ROOT}")


def find_npy_dir(base: Path) -> Path:
    """폴더가 한 겹 더 중첩된 경우까지 훑어서 .npy 가 실제로 있는 폴더를 찾는다."""
    if any(base.glob("*.npy")):
        return base
    for sub in sorted(d for d in base.iterdir() if d.is_dir()):
        if any(sub.glob("*.npy")):
            return sub
    raise FileNotFoundError(f".npy 파일이 없다: {base}")


TRAIN_DIR = find_npy_dir(LOCAL_DATA / "train")
VAL_DIR = find_npy_dir(LOCAL_DATA / "val")
TEST_LABEL_DIR = find_npy_dir(LOCAL_DATA / "test_label")
TEST_NOISY_DIR = find_npy_dir(LOCAL_DATA / "test_noise_only")

for name, d in [("train", TRAIN_DIR), ("val", VAL_DIR),
                ("test_label", TEST_LABEL_DIR), ("test_noise_only", TEST_NOISY_DIR)]:
    print(f"  {name:16s} {len(list(d.glob('*.npy'))):5d} files   {d}")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU 로)")


Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
  train             7268 files   /content/work/train
  val                100 files   /content/work/val
  test_label         100 files   /content/work/test_label
  test_noise_only    100 files   /content/work/test_noise_only/test_noise_only
GPU: NVIDIA A100-SXM4-80GB


## 2. Config

In [10]:
import json
import math
import random
import time
import zlib
from dataclasses import dataclass, field, asdict

import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

RUN_DIR = DRIVE_ROOT / "logs_restormer"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# 기존 DnCNN 노트북과 동일한 잡음 범위. 비교 공정성을 위해 바꾸지 않는다.
NOISE_RANGES: dict[str, tuple[float, float]] = {
    "gaussian": (0.0, 0.1),
    "rician": (0.0, 0.15),
    "uniform": (0.0, 0.2),
    "salt_and_pepper": (0.0, 0.2),
}


@dataclass
class RestormerConfig:
    """모델 구조. dim 과 num_blocks 를 키우면 성능은 오르고 학습은 느려진다."""
    in_channels: int = 1
    out_channels: int = 1
    dim: int = 48                                   # 첫 단계의 채널 수
    num_blocks: tuple = (2, 3, 3, 4)                # 단계별 트랜스포머 블록 개수
    num_refinement_blocks: int = 2
    heads: tuple = (1, 2, 4, 8)                     # 단계별 어텐션 갈래 수
    ffn_expansion: float = 2.66
    bias: bool = False


@dataclass
class TrainConfig:
    epochs: int = 60
    batch: int = 48
    patch: int = 128            # 학습 시 잘라 쓰는 크기 (8의 배수여야 함)
    lr: float = 5e-4
    lr_min: float = 1e-6
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    num_workers: int = 8
    val_batch: int = 8
    amp: bool = True            # bfloat16 혼합정밀도 (A100 권장)
    seed: int = 0


model_cfg = RestormerConfig()
train_cfg = TrainConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(train_cfg.seed)
np.random.seed(train_cfg.seed)
torch.manual_seed(train_cfg.seed)

print("run dir:", RUN_DIR)
print("device :", device)
for k, v in asdict(train_cfg).items():
    print(f"  {k}: {v}")


run dir: /content/drive/MyDrive/project5/logs_restormer
device : cuda
  epochs: 60
  batch: 48
  patch: 128
  lr: 0.0005
  lr_min: 1e-06
  weight_decay: 0.0001
  grad_clip: 1.0
  num_workers: 8
  val_batch: 8
  amp: True
  seed: 0


## 3. PSNR / SSIM  — 기존 노트북과 동일 구현

In [11]:
IMG_DIM: int = 4


class SSIMcal(torch.nn.Module):
    def __init__(self, win_size: int = 11, k1: float = 0.01, k2: float = 0.03):
        super().__init__()
        self.win_size = win_size
        self.k1, self.k2 = k1, k2
        self.register_buffer("w", torch.ones(1, 1, win_size, win_size) / win_size**2)
        np_ = win_size**2
        self.cov_norm = np_ / (np_ - 1)

    def forward(self, img: Tensor, ref: Tensor, data_range: Tensor) -> Tensor:
        data_range = data_range[:, None, None, None]
        C1 = (self.k1 * data_range) ** 2
        C2 = (self.k2 * data_range) ** 2

        w = self.w.to(img.device)
        ux = F.conv2d(img, w)
        uy = F.conv2d(ref, w)
        uxx = F.conv2d(img * img, w)
        uyy = F.conv2d(ref * ref, w)
        uxy = F.conv2d(img * ref, w)

        vx = self.cov_norm * (uxx - ux * ux)
        vy = self.cov_norm * (uyy - uy * uy)
        vxy = self.cov_norm * (uxy - ux * uy)

        A1 = 2 * ux * uy + C1
        A2 = 2 * vxy + C2
        B1 = ux**2 + uy**2 + C1
        B2 = vx + vy + C2
        return torch.mean((A1 * A2) / (B1 * B2), dim=[2, 3], keepdim=True)


ssim_cal = SSIMcal()


def calculate_ssim(img: Tensor, ref: Tensor) -> Tensor:
    if not (img.dim() == IMG_DIM and ref.dim() == IMG_DIM):
        raise ValueError("All tensors must be 4D.")
    ones = torch.ones(ref.shape[0], device=ref.device)
    return ssim_cal.forward(img, ref, ones)


def calculate_psnr(img: Tensor, ref: Tensor) -> Tensor:
    if not (img.dim() == IMG_DIM and ref.dim() == IMG_DIM):
        raise ValueError("All tensors must be 4D.")
    mse = torch.mean(F.mse_loss(img, ref, reduction="none"), dim=(1, 2, 3), keepdim=True)
    img_max = torch.amax(ref, dim=(1, 2, 3), keepdim=True)
    return 10 * torch.log10(img_max**2 / (mse + 1e-12))


## 4. 잡음 생성기 — 기존 노트북과 동일 구현

In [12]:
def gaussian_noise(img: Tensor, sigma: float) -> Tensor:
    return img + torch.randn_like(img) * sigma


def rician_noise(img: Tensor, sigma: float) -> Tensor:
    nr = torch.randn_like(img) * sigma
    ni = torch.randn_like(img) * sigma
    return torch.abs(img + nr + 1j * ni)


def uniform_noise(img: Tensor, sigma: float) -> Tensor:
    return img + (torch.rand_like(img) * 2.0 - 1.0) * sigma


def salt_and_pepper_noise(img: Tensor, sigma: float) -> Tensor:
    out = img.clone()
    total = img.numel()
    num = int(total * sigma / 2)
    coords = [torch.randint(0, d, (num,)) for d in img.shape]
    out[tuple(coords)] = img.max()
    coords = [torch.randint(0, d, (num,)) for d in img.shape]
    out[tuple(coords)] = 0
    return out


NOISE_FUNCS = {
    "gaussian": gaussian_noise,
    "rician": rician_noise,
    "uniform": uniform_noise,
    "salt_and_pepper": salt_and_pepper_noise,
}


class RandomNoiseSimulator:
    """4종 중 하나를 무작위로 골라, 해당 범위에서 sigma 를 뽑아 적용한다."""

    def __init__(self, noise_ranges: dict[str, tuple[float, float]] | None = None):
        self.noise_ranges = dict(noise_ranges or NOISE_RANGES)
        self.names = list(self.noise_ranges)

    def _sample(self, rng) -> tuple[str, float]:
        name = rng.choice(self.names)
        low, high = self.noise_ranges[name]
        return name, rng.uniform(low, high)

    def __call__(self, img: Tensor, seed: int | None = None) -> Tensor:
        if seed is None:
            name, sigma = self._sample(random)
            return NOISE_FUNCS[name](img, sigma)

        # 검증용: 파일마다 항상 같은 잡음이 나오도록 고정
        name, sigma = self._sample(random.Random(seed))
        state = torch.random.get_rng_state()
        torch.manual_seed(seed)
        try:
            out = NOISE_FUNCS[name](img, sigma)
        finally:
            torch.random.set_rng_state(state)
        return out


## 5. 데이터셋

- **train** : 매번 새 잡음을 얹고, 128x128 로 잘라 쓰고, 뒤집기/회전으로 늘린다
- **val**   : 파일명으로 잡음을 고정하고, 256x256 전체를 쓴다
- **test**  : 주어진 잡음 영상 파일을 그대로 쓴다 (필터 없음)


In [13]:
def load_npy(path: Path) -> Tensor:
    img = torch.from_numpy(np.load(str(path))).float()
    if img.dim() == 2:
        img = img.unsqueeze(0)
    return img


class DenoiseDataset(Dataset):
    def __init__(
        self,
        clean_dir: Path,
        mode: str,                       # "train" | "val" | "test"
        noisy_dir: Path | None = None,   # mode == "test" 일 때만 사용
        patch: int | None = None,
    ):
        self.files = sorted(clean_dir.glob("*.npy"))
        if not self.files:
            raise FileNotFoundError(f"no .npy in {clean_dir}")
        self.mode = mode
        self.noisy_dir = noisy_dir
        self.patch = patch
        self.sim = RandomNoiseSimulator()

    def __len__(self) -> int:
        return len(self.files)

    def _augment(self, label: Tensor) -> Tensor:
        if self.patch is not None:
            _, h, w = label.shape
            p = self.patch
            i = random.randint(0, h - p)
            j = random.randint(0, w - p)
            label = label[:, i:i + p, j:j + p]
        if random.random() < 0.5:
            label = torch.flip(label, dims=[1])
        if random.random() < 0.5:
            label = torch.flip(label, dims=[2])
        k = random.randint(0, 3)
        if k:
            label = torch.rot90(label, k, dims=[1, 2])
        return label.contiguous()

    def __getitem__(self, idx: int):
        path = self.files[idx]
        name = path.name
        label = load_npy(path)

        if self.mode == "train":
            label = self._augment(label)      # 자르기/뒤집기 먼저, 잡음은 그 뒤에
            noisy = self.sim(label)
        elif self.mode == "val":
            noisy = self.sim(label, seed=zlib.crc32(name.encode()))
        else:
            noisy_path = self.noisy_dir / name
            if not noisy_path.exists():
                raise FileNotFoundError(noisy_path)
            noisy = load_npy(noisy_path)

        return label, noisy, name


train_set = DenoiseDataset(TRAIN_DIR, "train", patch=train_cfg.patch)
val_set = DenoiseDataset(VAL_DIR, "val")
test_set = DenoiseDataset(TEST_LABEL_DIR, "test", noisy_dir=TEST_NOISY_DIR)

train_loader = DataLoader(train_set, batch_size=train_cfg.batch, shuffle=True,
                          num_workers=train_cfg.num_workers, pin_memory=True, drop_last=True,
                          persistent_workers=train_cfg.num_workers > 0)
val_loader = DataLoader(val_set, batch_size=train_cfg.val_batch, shuffle=False,
                        num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=train_cfg.val_batch, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f"train {len(train_set)} / val {len(val_set)} / test {len(test_set)}")
_l, _n, _nm = train_set[0]
print("sample:", _l.shape, _n.shape, _nm)


train 7268 / val 100 / test 100
sample: torch.Size([1, 128, 128]) torch.Size([1, 128, 128]) L1_0009994b978735c76dba161531a98aed.npy


## 6. Restormer 모델

세 부분으로 나뉜다.

1. **채널 방향 어텐션 (MDTA, Multi-Dconv head Transposed Attention)**
   픽셀끼리 짝지어 비교하면 256x256 이미지에서 약 43억 번의 비교가 필요해 불가능하다.
   그래서 픽셀이 아니라 **채널끼리** 비교한다. 각 채널을 계산할 때 이미지 전체의 픽셀을
   사용하므로 "멀리 떨어진 곳을 참조한다"는 목적은 유지하면서 계산량만 줄어든다.

2. **게이트 방식 전방 전달망 (GDFN, Gated-Dconv Feed-Forward Network)**
   신호를 두 갈래로 나눠 한쪽이 "얼마나 통과시킬지"를 정하고 다른 쪽과 곱한다.
   잡음처럼 다음 층에 넘길 가치가 없는 성분을 여기서 막는다.

3. **깊이별 합성곱 (depthwise convolution)**
   채널 방향 어텐션만 쓰면 "어느 픽셀이 어디 있는지"라는 공간 정보가 약해지므로,
   채널마다 따로 합성곱을 걸어 가까운 이웃 정보를 보충한다.

전체 구조는 U자 형태다. 해상도를 세 번 줄였다가 다시 세 번 키우고, 같은 해상도의
앞쪽 결과를 뒤쪽에 이어붙여 잃어버린 세부를 되살린다. 세 번 줄이므로 입력의 가로세로는
**8의 배수**여야 한다 (256, 128 모두 만족).


In [14]:
class LayerNorm2d(nn.Module):
    """채널 방향으로 정규화한다."""

    def __init__(self, dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x: Tensor) -> Tensor:
        mu = x.mean(dim=1, keepdim=True)
        var = x.var(dim=1, keepdim=True, unbiased=False)
        x = (x - mu) / torch.sqrt(var + 1e-5)
        return x * self.weight[None, :, None, None] + self.bias[None, :, None, None]


class MDTA(nn.Module):
    """채널 방향 어텐션."""

    def __init__(self, dim: int, heads: int, bias: bool):
        super().__init__()
        self.heads = heads
        self.temperature = nn.Parameter(torch.ones(heads, 1, 1))
        self.qkv = nn.Conv2d(dim, dim * 3, 1, bias=bias)
        self.qkv_dw = nn.Conv2d(dim * 3, dim * 3, 3, padding=1, groups=dim * 3, bias=bias)
        self.proj = nn.Conv2d(dim, dim, 1, bias=bias)

    def forward(self, x: Tensor) -> Tensor:
        b, c, h, w = x.shape
        q, k, v = self.qkv_dw(self.qkv(x)).chunk(3, dim=1)

        # (B, heads, C/heads, H*W) 로 편다
        def split(t: Tensor) -> Tensor:
            return t.reshape(b, self.heads, c // self.heads, h * w)

        q, k, v = split(q), split(k), split(v)
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        attn = (q @ k.transpose(-2, -1)) * self.temperature   # (B, heads, C/h, C/h)
        attn = attn.softmax(dim=-1)
        out = attn @ v                                        # (B, heads, C/h, H*W)
        return self.proj(out.reshape(b, c, h, w))


class GDFN(nn.Module):
    """게이트 방식 전방 전달망."""

    def __init__(self, dim: int, expansion: float, bias: bool):
        super().__init__()
        hidden = int(dim * expansion)
        self.project_in = nn.Conv2d(dim, hidden * 2, 1, bias=bias)
        self.dw = nn.Conv2d(hidden * 2, hidden * 2, 3, padding=1, groups=hidden * 2, bias=bias)
        self.project_out = nn.Conv2d(hidden, dim, 1, bias=bias)

    def forward(self, x: Tensor) -> Tensor:
        x1, x2 = self.dw(self.project_in(x)).chunk(2, dim=1)
        return self.project_out(F.gelu(x1) * x2)              # x2 가 통과량을 정하는 문 역할


class TransformerBlock(nn.Module):
    def __init__(self, dim: int, heads: int, expansion: float, bias: bool):
        super().__init__()
        self.norm1 = LayerNorm2d(dim)
        self.attn = MDTA(dim, heads, bias)
        self.norm2 = LayerNorm2d(dim)
        self.ffn = GDFN(dim, expansion, bias)

    def forward(self, x: Tensor) -> Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class Downsample(nn.Module):
    """가로세로 절반, 채널 두 배."""

    def __init__(self, dim: int):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(dim, dim // 2, 3, 1, 1, bias=False),
                                  nn.PixelUnshuffle(2))

    def forward(self, x: Tensor) -> Tensor:
        return self.body(x)


class Upsample(nn.Module):
    """가로세로 두 배, 채널 절반."""

    def __init__(self, dim: int):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(dim, dim * 2, 3, 1, 1, bias=False),
                                  nn.PixelShuffle(2))

    def forward(self, x: Tensor) -> Tensor:
        return self.body(x)


class Restormer(nn.Module):
    def __init__(self, cfg: RestormerConfig):
        super().__init__()
        d = cfg.dim
        e, bias = cfg.ffn_expansion, cfg.bias
        nb, nh = cfg.num_blocks, cfg.heads

        def stage(dim: int, heads: int, n: int) -> nn.Sequential:
            return nn.Sequential(*[TransformerBlock(dim, heads, e, bias) for _ in range(n)])

        self.patch_embed = nn.Conv2d(cfg.in_channels, d, 3, 1, 1, bias=bias)

        self.enc1 = stage(d, nh[0], nb[0])
        self.down1 = Downsample(d)
        self.enc2 = stage(d * 2, nh[1], nb[1])
        self.down2 = Downsample(d * 2)
        self.enc3 = stage(d * 4, nh[2], nb[2])
        self.down3 = Downsample(d * 4)
        self.latent = stage(d * 8, nh[3], nb[3])

        self.up3 = Upsample(d * 8)
        self.reduce3 = nn.Conv2d(d * 8, d * 4, 1, bias=bias)
        self.dec3 = stage(d * 4, nh[2], nb[2])

        self.up2 = Upsample(d * 4)
        self.reduce2 = nn.Conv2d(d * 4, d * 2, 1, bias=bias)
        self.dec2 = stage(d * 2, nh[1], nb[1])

        self.up1 = Upsample(d * 2)
        self.dec1 = stage(d * 2, nh[0], nb[0])          # 이어붙여 채널이 2d 가 된다
        self.refine = stage(d * 2, nh[0], cfg.num_refinement_blocks)
        self.output = nn.Conv2d(d * 2, cfg.out_channels, 3, 1, 1, bias=bias)

    def forward(self, x: Tensor) -> Tensor:
        f1 = self.enc1(self.patch_embed(x))
        f2 = self.enc2(self.down1(f1))
        f3 = self.enc3(self.down2(f2))
        f4 = self.latent(self.down3(f3))

        y = self.dec3(self.reduce3(torch.cat([self.up3(f4), f3], dim=1)))
        y = self.dec2(self.reduce2(torch.cat([self.up2(y), f2], dim=1)))
        y = self.dec1(torch.cat([self.up1(y), f1], dim=1))
        y = self.refine(y)
        return self.output(y) + x                        # 잔차 학습: 빼야 할 양만 배운다


net = Restormer(model_cfg).to(device)
n_param = sum(p.numel() for p in net.parameters())
print(f"parameters: {n_param/1e6:.2f} M")

with torch.no_grad():
    print("shape check:", net(torch.randn(1, 1, 256, 256, device=device)).shape)


parameters: 14.96 M
shape check: torch.Size([1, 1, 256, 256])


In [15]:
# 이전 학습 기록 삭제 (처음부터 다시 학습하기 위해)
LOG_DIR = DRIVE_ROOT / "logs_restormer"
for f in ["checkpoint_last.ckpt", "checkpoint_best.ckpt", "history.json"]:
    (LOG_DIR / f).unlink(missing_ok=True)
print("이전 기록 삭제 완료")

이전 기록 삭제 완료


## 7. 학습

- 손실 함수는 **L1**(절댓값 차이). 영상 복원에서 L2 보다 결과가 선명한 경우가 많다.
- **bfloat16 혼합정밀도**로 계산량을 줄인다 (A100 에서 안전하다).
- 매 epoch 마지막 상태를 Drive 에 저장하므로, 세션이 끊겨도 이어서 학습할 수 있다.


In [16]:
CKPT_LAST = RUN_DIR / "checkpoint_last.ckpt"
CKPT_BEST = RUN_DIR / "checkpoint_best.ckpt"
HISTORY = RUN_DIR / "history.json"

optimizer = torch.optim.AdamW(net.parameters(), lr=train_cfg.lr,
                              betas=(0.9, 0.999), weight_decay=train_cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=train_cfg.epochs, eta_min=train_cfg.lr_min)
loss_func = nn.L1Loss()

amp_dtype = torch.bfloat16 if train_cfg.amp else None


def run_train_epoch(epoch: int) -> float:
    net.train()
    total, count = 0.0, 0
    pbar = tqdm(train_loader, desc=f"train ep {epoch+1}/{train_cfg.epochs}", leave=False)
    for label, noisy, _ in pbar:
        label = label.to(device, non_blocking=True)
        noisy = noisy.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        if amp_dtype is not None:
            with torch.autocast("cuda", dtype=amp_dtype):
                loss = loss_func(net(noisy), label)
        else:
            loss = loss_func(net(noisy), label)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), train_cfg.grad_clip)
        optimizer.step()

        total += loss.item() * label.shape[0]
        count += label.shape[0]
        pbar.set_postfix(loss=f"{total/count:.5f}")
    return total / count


@torch.no_grad()
def evaluate(loader: DataLoader, desc: str) -> tuple[float, float]:
    net.eval()
    psnr_list, ssim_list = [], []
    for label, noisy, _ in tqdm(loader, desc=desc, leave=False):
        label = label.to(device, non_blocking=True)
        noisy = noisy.to(device, non_blocking=True)
        if amp_dtype is not None:
            with torch.autocast("cuda", dtype=amp_dtype):
                out = net(noisy)
            out = out.float()
        else:
            out = net(noisy)
        psnr_list.append(calculate_psnr(out, label).flatten())
        ssim_list.append(calculate_ssim(out, label).flatten())
    return (torch.cat(psnr_list).mean().item(), torch.cat(ssim_list).mean().item())


def save_ckpt(path: Path, epoch: int, best_psnr: float) -> None:
    torch.save({
        "epoch": epoch,
        "best_psnr": best_psnr,
        "model_state_dict": net.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "model_config": asdict(model_cfg),
    }, path)


start_epoch, best_psnr, history = 0, -1.0, []
if CKPT_LAST.exists():
    ck = torch.load(CKPT_LAST, map_location=device, weights_only=False)
    net.load_state_dict(ck["model_state_dict"])
    optimizer.load_state_dict(ck["optimizer_state_dict"])
    scheduler.load_state_dict(ck["scheduler_state_dict"])
    start_epoch = ck["epoch"] + 1
    best_psnr = ck["best_psnr"]
    if HISTORY.exists():
        history = json.loads(HISTORY.read_text())
    print(f"이어서 학습: epoch {start_epoch} 부터, 현재 best val PSNR {best_psnr:.3f}")
else:
    print("처음부터 학습")


처음부터 학습


In [ ]:
t0 = time.time()
for epoch in range(start_epoch, train_cfg.epochs):
    train_loss = run_train_epoch(epoch)
    val_psnr, val_ssim = evaluate(val_loader, "valid")
    scheduler.step()

    is_best = val_psnr > best_psnr
    if is_best:
        best_psnr = val_psnr
        save_ckpt(CKPT_BEST, epoch, best_psnr)

    save_ckpt(CKPT_LAST, epoch, best_psnr)
    history.append({"epoch": epoch + 1, "loss": train_loss,
                    "val_psnr": val_psnr, "val_ssim": val_ssim,
                    "lr": optimizer.param_groups[0]["lr"]})
    HISTORY.write_text(json.dumps(history, indent=1))

    mark = "  <- best" if is_best else ""
    elapsed = (time.time() - t0) / 60
    print(f"ep {epoch+1:3d}/{train_cfg.epochs}  loss {train_loss:.5f}  "
          f"val PSNR {val_psnr:7.3f}  SSIM {val_ssim:.4f}  [{elapsed:.1f} min]{mark}")

print(f"\n학습 종료. best val PSNR = {best_psnr:.3f}")
print(f"best checkpoint: {CKPT_BEST}")


train ep 1/60:   0%|          | 0/151 [00:00<?, ?it/s]

valid:   0%|          | 0/13 [00:00<?, ?it/s]

ep   1/60  loss 0.03976  val PSNR  28.537  SSIM 0.7814  [1.9 min]  <- best


train ep 2/60:   0%|          | 0/151 [00:00<?, ?it/s]

valid:   0%|          | 0/13 [00:00<?, ?it/s]

...

ep  16/60  loss 0.01470  val PSNR  34.736  SSIM 0.9117  [30.3 min]  <- best


train ep 17/60:   0%|          | 0/151 [00:00<?, ?it/s]

valid:   0%|          | 0/13 [00:00<?, ?it/s]

ep  17/60  loss 0.01476  val PSNR  34.719  SSIM 0.9149  [32.2 min]


train ep 18/60:   0%|          | 0/151 [00:00<?, ?it/s]

## 8. 학습 곡선

In [ ]:
import matplotlib.pyplot as plt

hist = json.loads(HISTORY.read_text())
ep = [h["epoch"] for h in hist]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(ep, [h["loss"] for h in hist]);      axes[0].set_title("train L1 loss")
axes[1].plot(ep, [h["val_psnr"] for h in hist]);  axes[1].set_title("val PSNR (dB)")
axes[2].plot(ep, [h["val_ssim"] for h in hist]);  axes[2].set_title("val SSIM")
for ax in axes:
    ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
axes[1].axhline(34.415, ls="--", c="r", label="팀원 B (test ALL)")
axes[1].legend()
fig.tight_layout(); plt.show()


## 9. 테스트 — 팀원 표와 같은 형식

`test_noise_only` 의 잡음 영상을 **아무 필터도 거치지 않고** 그대로 넣는다.
비교용 고전 필터(평균 / 중앙값 / 적응)도 같은 입력에 적용해 함께 싣는다.


In [ ]:
# 비교용 고전 필터 (기존 test 노트북과 동일)
def mean_filter(x: Tensor, k: int = 3) -> Tensor:
    pad = k // 2
    return F.avg_pool2d(F.pad(x, (pad,) * 4, mode="reflect"), kernel_size=k, stride=1)


def median_filter(x: Tensor, k: int = 3) -> Tensor:
    pad = k // 2
    xp = F.pad(x, (pad,) * 4, mode="reflect")
    patches = xp.unfold(2, k, 1).unfold(3, k, 1)
    return patches.reshape(*patches.shape[:4], -1).median(dim=-1).values


def adaptive_filter(x: Tensor, k: int = 5) -> Tensor:
    pad = k // 2
    xp = F.pad(x, (pad,) * 4, mode="reflect")
    local_mean = F.avg_pool2d(xp, kernel_size=k, stride=1)
    local_sq = F.avg_pool2d(xp.pow(2), kernel_size=k, stride=1)
    local_var = (local_sq - local_mean.pow(2)).clamp_min(0.0)
    noise_var = local_var.flatten(2).median(dim=-1).values[:, :, None, None]
    ratio = (noise_var / local_var.clamp_min(1e-8)).clamp(max=1.0)
    return x - ratio * (x - local_mean)


# 파일별 잡음 종류
meta_path = TEST_NOISY_DIR / "noise_meta.json"
noise_lookup: dict[str, str] = {}
if meta_path.exists():
    for item in json.loads(meta_path.read_text()):
        noise_lookup[item["file"]] = item["noise_type"]
else:
    print("noise_meta.json 이 없다. 종류별 분리 없이 전체 평균만 출력된다.")


In [ ]:
best = torch.load(CKPT_BEST, map_location=device, weights_only=False)
net.load_state_dict(best["model_state_dict"])
net.eval()
print(f"loaded best checkpoint (epoch {best['epoch']+1}, val PSNR {best['best_psnr']:.3f})")

METHODS = ["noisy", "restormer", "mean", "median", "adaptive"]
LABELS = {"noisy": "Noisy (input)", "restormer": "Restormer",
          "mean": "Mean 3x3", "median": "Median 3x3", "adaptive": "Adaptive 5x5"}

rows: list[dict] = []
samples: dict[str, dict] = {}

with torch.no_grad():
    for label, noisy, names in tqdm(test_loader, desc="test"):
        label = label.to(device)
        noisy = noisy.to(device)
        if amp_dtype is not None:
            with torch.autocast("cuda", dtype=amp_dtype):
                pred = net(noisy)
            pred = pred.float()
        else:
            pred = net(noisy)

        outs = {"noisy": noisy, "restormer": pred,
                "mean": mean_filter(noisy), "median": median_filter(noisy),
                "adaptive": adaptive_filter(noisy)}

        for i in range(label.shape[0]):
            ref = label[i:i + 1]
            row = {"file": names[i], "noise_type": noise_lookup.get(names[i], "unknown")}
            for m in METHODS:
                p = outs[m][i:i + 1]
                row[f"psnr_{m}"] = calculate_psnr(p, ref).item()
                row[f"ssim_{m}"] = calculate_ssim(p, ref).item()
            rows.append(row)

            if row["noise_type"] not in samples:
                s = {"metrics": row, "label": ref.cpu().numpy().squeeze()}
                for m in METHODS:
                    s[m] = outs[m][i:i + 1].cpu().numpy().squeeze()
                samples[row["noise_type"]] = s

(RUN_DIR / "test_metrics.json").write_text(json.dumps(rows, indent=1))
print("saved:", RUN_DIR / "test_metrics.json")


In [ ]:
TEAM_B = {
    "psnr": {"gaussian": 34.413, "rician": 30.529, "uniform": 34.213,
             "salt_and_pepper": 38.505, "ALL": 34.415},
    "ssim": {"gaussian": 0.9180, "rician": 0.8900, "uniform": 0.9477,
             "salt_and_pepper": 0.9869, "ALL": 0.9356},
}
ORDER = [*NOISE_RANGES.keys(), "unknown"]


def print_table(metric: str, width: int = 15) -> None:
    fmt = ".3f" if metric == "psnr" else ".4f"
    print(f"[{metric.upper()}]")
    header = f"{'noise':<18}{'n':>5}" + "".join(f"{LABELS[m]:>{width}}" for m in METHODS)
    header += f"{'팀원B DnCNN':>15}{'차이':>10}"
    print(header)
    print("-" * len(header))

    def line(name: str, sub: list[dict]) -> None:
        cells = "".join(f"{np.mean([r[f'{metric}_{m}'] for r in sub]):>{width}{fmt}}" for m in METHODS)
        ref = TEAM_B[metric].get(name)
        mine = float(np.mean([r[f"{metric}_restormer"] for r in sub]))
        extra = f"{ref:>15{fmt}}{mine-ref:>+10{fmt}}" if ref is not None else " " * 25
        print(f"{name:<18}{len(sub):>5}{cells}{extra}")

    for nz in ORDER:
        sub = [r for r in rows if r["noise_type"] == nz]
        if sub:
            line(nz, sub)
    print("-" * len(header))
    line("ALL", rows)
    print()


print_table("psnr")
print_table("ssim")


## 10. 잡음 종류별 결과 이미지

In [ ]:
cols = ["noisy", "restormer", "mean", "median", "adaptive"]
order = [nz for nz in ORDER if nz in samples]

fig, axes = plt.subplots(len(order), len(cols) + 1,
                         figsize=(3.2 * (len(cols) + 1), 3.4 * len(order)))
axes = np.asarray(axes).reshape(len(order), len(cols) + 1)

for r, nz in enumerate(order):
    s = samples[nz]
    vmax = float(np.percentile(s["label"], 98) * 1.2)

    axes[r, 0].imshow(s["label"], cmap="gray", vmin=0, vmax=vmax)
    axes[r, 0].set_ylabel(nz, fontsize=11)
    axes[r, 0].set_title("label (원본 영상)", fontsize=9)
    axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])

    for c, m in enumerate(cols, start=1):
        axes[r, c].imshow(s[m], cmap="gray", vmin=0, vmax=vmax)
        axes[r, c].set_title(
            f"{LABELS[m]}\nPSNR {s['metrics'][f'psnr_{m}']:.2f} / SSIM {s['metrics'][f'ssim_{m}']:.4f}",
            fontsize=9)
        axes[r, c].axis("off")

fig.suptitle("Restormer (필터 없음) vs 고전 필터", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(RUN_DIR / "test_grid.png", dpi=150, bbox_inches="tight")
plt.show()
